# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from config import *
from utils import *

## 1 Nettoyer la colonne MIN dans les boxscores

In [5]:



# Si tu utilises plusieurs fichiers, adapte avec concat, ici sur un seul batch pour la démo :
boxscores_file = get_latest_file(DATA_BOXSCORES_DIR)  # Ou DATA_PLAYERS_DIR selon ton code
games_file = get_latest_file(DATA_GAMES_DIR)  # Ou DATA_PLAYERS_DIR selon ton code

df_boxscores = pd.read_csv(boxscores_file, low_memory=False)
df_games = pd.read_csv(games_file, low_memory=False)

print(df_games.columns)

# 3. Merge pour ajouter GAME_DATE à chaque ligne de boxscore
if 'GAME_DATE' not in df_boxscores.columns:
    # Pour éviter les doublons ou les merges en cascade si tu relances le code
    if 'GAME_DATE' in df_games.columns:
        df_boxscores = df_boxscores.merge(
            df_games[['GAME_ID', 'GAME_DATE']],
            on='GAME_ID',
            how='left'
        )
        # Conversion en datetime
        df_boxscores['GAME_DATE'] = pd.to_datetime(df_boxscores['GAME_DATE'])
    else:
        print("No GAME_DATE column found in games_df. Please check your merge operation.")

# Fonction de conversion “mm:ss” -> float minutes
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    if isinstance(val, float) or isinstance(val, int):
        return float(val)
    try:
        parts = str(val).split(':')
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return minutes + seconds/60
        else:
            # Cas où c'est déjà un float/int
            return float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['MIN'].apply(convert_minutes)

print(df_boxscores[['PLAYER_NAME', 'MIN', 'MINUTES_PLAYED']].head(10))
print(df_boxscores['MINUTES_PLAYED'].describe())


Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON'],
      dtype='object')
        PLAYER_NAME    MIN  MINUTES_PLAYED
0  Dante Cunningham  26:31       26.516667
1  Dante Cunningham  26:31       26.516667
2     Anthony Davis  34:31       34.516667
3     Anthony Davis  34:31       34.516667
4  Kendrick Perkins  15:53       15.883333
5  Kendrick Perkins  15:53       15.883333
6       Eric Gordon  33:29       33.483333
7       Eric Gordon  33:29       33.483333
8     Nate Robinson  19:03       19.050000
9     Nate Robinson  19:03       19.050000
count    118258.000000
mean         18.842910
std          12.845809
min           0.000000
25%           6.800000
50%          20.116667
75%          29.566667
max          60.116667
Name: MINUTES_PLAYED, d

In [6]:
df_boxscores

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,GAME_DATE,MINUTES_PLAYED
0,21500003,1610612740,NOP,New Orleans,201967,Dante Cunningham,Dante,G,NaN,26:31,...,2.0,0.0,0.0,0.0,0.0,2.0,13.0,-15.0,2015-10-27,26.516667
1,21500003,1610612740,NOP,New Orleans,201967,Dante Cunningham,Dante,G,NaN,26:31,...,2.0,0.0,0.0,0.0,0.0,2.0,13.0,-15.0,2015-10-27,26.516667
2,21500003,1610612740,NOP,New Orleans,203076,Anthony Davis,Anthony,G,NaN,34:31,...,6.0,2.0,0.0,3.0,5.0,4.0,18.0,-16.0,2015-10-27,34.516667
3,21500003,1610612740,NOP,New Orleans,203076,Anthony Davis,Anthony,G,NaN,34:31,...,6.0,2.0,0.0,3.0,5.0,4.0,18.0,-16.0,2015-10-27,34.516667
4,21500003,1610612740,NOP,New Orleans,2570,Kendrick Perkins,Kendrick,C,NaN,15:53,...,4.0,1.0,0.0,0.0,0.0,1.0,10.0,-12.0,2015-10-27,15.883333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118253,21600989,1610612745,HOU,Houston,201628,Bobby Brown,Bobby,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-03-12,0.000000
118254,21600989,1610612745,HOU,Houston,1626149,Montrezl Harrell,Montrezl,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-03-12,0.000000
118255,21600989,1610612745,HOU,Houston,1626149,Montrezl Harrell,Montrezl,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-03-12,0.000000
118256,21600989,1610612745,HOU,Houston,1627787,Kyle Wiltjer,Kyle,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-03-12,0.000000


## Type correction

In [7]:
cols_to_float = [
    'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 
    'FTM', 'FTA', 'OREB', 'DREB', 'PLUS_MINUS'
]
for col in cols_to_float:
    if col in df_boxscores.columns:
        df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)


 ### Optionnel : repérage/ajout d’un flag “Starter”

In [13]:
df_boxscores['IS_STARTER'] = df_boxscores['START_POSITION'].notna() & (df_boxscores['START_POSITION'] != '')

## Étape 2 — Agrégation de stats par équipe et par match

In [18]:
agg_cols = [
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED'
    ]


team_match_stats = df_boxscores.groupby(['GAME_ID', 'TEAM_ID', 'GAME_DATE'])[agg_cols].sum().reset_index()

# Optionnel : rajoute l’équipe adverse dans chaque ligne pour faciliter le merge futur
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1
)



In [20]:
team_match_stats


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID
0,21500001,1610612737,2015-10-27,74.0,164.0,9.174,16.0,54.0,5.198,24.0,...,80.0,44.0,18.0,8.0,30.0,50.0,188.0,-120.0,480.000000,1610612765
1,21500001,1610612765,2015-10-27,74.0,192.0,6.816,24.0,58.0,4.708,40.0,...,118.0,46.0,10.0,6.0,30.0,30.0,212.0,120.0,480.000000,1610612737
2,21500002,1610612739,2015-10-27,76.0,188.0,6.388,18.0,58.0,3.116,20.0,...,100.0,52.0,10.0,14.0,20.0,42.0,190.0,-20.0,480.000000,1610612741
3,21500002,1610612741,2015-10-27,74.0,174.0,7.932,14.0,38.0,4.300,32.0,...,94.0,26.0,12.0,20.0,26.0,44.0,194.0,20.0,480.000000,1610612739
4,21500003,1610612740,2015-10-27,70.0,166.0,8.344,12.0,36.0,3.572,38.0,...,66.0,42.0,18.0,6.0,36.0,52.0,190.0,-160.0,480.000000,1610612744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4205,41500405,1610612744,2016-06-13,128.0,352.0,10.524,56.0,172.0,7.180,76.0,...,172.0,72.0,24.0,36.0,68.0,84.0,388.0,-300.0,960.066667,1610612739
4206,41500406,1610612739,2016-06-16,160.0,308.0,22.412,40.0,108.0,7.200,100.0,...,180.0,96.0,48.0,28.0,40.0,100.0,460.0,280.0,960.000000,1610612744
4207,41500406,1610612744,2016-06-16,132.0,328.0,23.568,60.0,156.0,15.048,80.0,...,140.0,76.0,20.0,12.0,56.0,100.0,404.0,-280.0,960.000000,1610612739
4208,41500407,1610612739,2016-06-19,132.0,328.0,13.776,24.0,100.0,4.732,84.0,...,192.0,68.0,28.0,24.0,44.0,60.0,372.0,80.0,960.000000,1610612744


##  Renommer les colonnes pour merge

In [21]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]

df_team = team_match_stats.copy()
df_opp = team_match_stats.copy()
df_opp = df_opp.rename(
    columns={col: f"OPP_{col}" for col in team_cols}
).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'}
)

df_team
df_opp

,GAME_ID,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,...,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,TEAM_ID
0,21500001,1610612737,2015-10-27,74.0,164.0,9.174,16.0,54.0,5.198,24.0,...,80.0,44.0,18.0,8.0,30.0,50.0,188.0,-120.0,480.000000,1610612765
1,21500001,1610612765,2015-10-27,74.0,192.0,6.816,24.0,58.0,4.708,40.0,...,118.0,46.0,10.0,6.0,30.0,30.0,212.0,120.0,480.000000,1610612737
2,21500002,1610612739,2015-10-27,76.0,188.0,6.388,18.0,58.0,3.116,20.0,...,100.0,52.0,10.0,14.0,20.0,42.0,190.0,-20.0,480.000000,1610612741
3,21500002,1610612741,2015-10-27,74.0,174.0,7.932,14.0,38.0,4.300,32.0,...,94.0,26.0,12.0,20.0,26.0,44.0,194.0,20.0,480.000000,1610612739
4,21500003,1610612740,2015-10-27,70.0,166.0,8.344,12.0,36.0,3.572,38.0,...,66.0,42.0,18.0,6.0,36.0,52.0,190.0,-160.0,480.000000,1610612744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4205,41500405,1610612744,2016-06-13,128.0,352.0,10.524,56.0,172.0,7.180,76.0,...,172.0,72.0,24.0,36.0,68.0,84.0,388.0,-300.0,960.066667,1610612739
4206,41500406,1610612739,2016-06-16,160.0,308.0,22.412,40.0,108.0,7.200,100.0,...,180.0,96.0,48.0,28.0,40.0,100.0,460.0,280.0,960.000000,1610612744
4207,41500406,1610612744,2016-06-16,132.0,328.0,23.568,60.0,156.0,15.048,80.0,...,140.0,76.0,20.0,12.0,56.0,100.0,404.0,-280.0,960.000000,1610612739
4208,41500407,1610612739,2016-06-19,132.0,328.0,13.776,24.0,100.0,4.732,84.0,...,192.0,68.0,28.0,24.0,44.0,60.0,372.0,80.0,960.000000,1610612744


## Merge

In [22]:
match_dataset = pd.merge(
    df_team,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [23]:
match_dataset 
match_dataset.shape
match_dataset.head(10)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,21500001,1610612737,2015-10-27,74.0,164.0,9.174,16.0,54.0,5.198,24.0,...,72.0,118.0,46.0,10.0,6.0,30.0,30.0,212.0,120.0,480.0
1,21500001,1610612765,2015-10-27,74.0,192.0,6.816,24.0,58.0,4.708,40.0,...,66.0,80.0,44.0,18.0,8.0,30.0,50.0,188.0,-120.0,480.0
2,21500002,1610612739,2015-10-27,76.0,188.0,6.388,18.0,58.0,3.116,20.0,...,80.0,94.0,26.0,12.0,20.0,26.0,44.0,194.0,20.0,480.0
3,21500002,1610612741,2015-10-27,74.0,174.0,7.932,14.0,38.0,4.300,32.0,...,78.0,100.0,52.0,10.0,14.0,20.0,42.0,190.0,-20.0,480.0
4,21500003,1610612740,2015-10-27,70.0,166.0,8.344,12.0,36.0,3.572,38.0,...,70.0,112.0,58.0,16.0,14.0,40.0,58.0,222.0,160.0,480.0
5,21500003,1610612744,2015-10-27,82.0,192.0,8.264,18.0,60.0,4.034,40.0,...,50.0,66.0,42.0,18.0,6.0,36.0,52.0,190.0,-160.0,480.0
6,21500004,1610612753,2015-10-28,74.0,200.0,7.310,10.0,52.0,2.116,16.0,...,68.0,98.0,34.0,16.0,18.0,34.0,28.0,176.0,10.0,480.0
7,21500004,1610612764,2015-10-28,66.0,168.0,5.854,14.0,56.0,2.732,30.0,...,78.0,112.0,40.0,18.0,12.0,28.0,44.0,174.0,-10.0,480.0
8,21500005,1610612738,2015-10-28,78.0,170.0,9.072,16.0,48.0,4.334,52.0,...,64.0,92.0,24.0,22.0,12.0,44.0,44.0,190.0,-170.0,480.0
9,21500005,1610612755,2015-10-28,68.0,166.0,6.456,14.0,44.0,1.466,40.0,...,62.0,82.0,62.0,20.0,14.0,34.0,46.0,224.0,170.0,480.0


In [24]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED',
       'OPP_TEAM_ID', 'OPP_GAME_DATE', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
       'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTM', 'OPP_FTA',
       'OPP_FT_PCT', 'OPP_OREB', 'OPP_DREB', 'OPP_REB', 'OPP_AST', 'OPP_STL',
       'OPP_BLK', 'OPP_TO', 'OPP_PF', 'OPP_PTS', 'OPP_PLUS_MINUS',
       'OPP_MINUTES_PLAYED'],
      dtype='object')